# 한국어 데이터셋 구조 점검

다운로드한 `data/raw`의 Parquet·JSONL 파일을 메모리에 한꺼번에 올리지 않고 점검합니다.

확인 항목:
- 파일별 크기·행 수·컬럼
- 데이터 샘플과 중첩 구조
- 결측치와 컬럼 타입
- 대화 데이터의 role 분포
- 텍스트 길이 분포

In [1]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
SAMPLE_ROWS = 2_000

if not RAW_DIR.exists():
    raise FileNotFoundError(f'데이터 폴더가 없습니다: {RAW_DIR}')

print(f'프로젝트: {ROOT}')
print(f'데이터 폴더: {RAW_DIR}')

프로젝트: c:\Users\User\Desktop\trans
데이터 폴더: c:\Users\User\Desktop\trans\data\raw


## 1. 파일 인벤토리

In [2]:
def parquet_info(path):
    pf = pq.ParquetFile(path)
    return {
        'dataset': path.relative_to(RAW_DIR).parts[0],
        'file': path.relative_to(ROOT).as_posix(),
        'format': 'parquet',
        'rows': pf.metadata.num_rows,
        'size_mb': round(path.stat().st_size / 1024**2, 2),
        'columns': ', '.join(pf.schema_arrow.names),
    }

def jsonl_info(path):
    rows = 0
    keys = set()
    with path.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows += 1
                if rows <= 1000:
                    value = json.loads(line)
                    if isinstance(value, dict):
                        keys.update(value.keys())
    return {
        'dataset': path.relative_to(RAW_DIR).parts[0],
        'file': path.relative_to(ROOT).as_posix(),
        'format': 'jsonl',
        'rows': rows,
        'size_mb': round(path.stat().st_size / 1024**2, 2),
        'columns': ', '.join(sorted(keys)),
    }

files = sorted(p for p in RAW_DIR.rglob('*') if p.is_file() and '.cache' not in p.parts)
inventory = []
for path in files:
    if path.suffix == '.parquet':
        inventory.append(parquet_info(path))
    elif path.suffix == '.jsonl':
        inventory.append(jsonl_info(path))

inventory_df = pd.DataFrame(inventory)
display(inventory_df)
print(f'파일 수: {len(inventory_df)}, 총 행 수: {inventory_df.rows.sum():,}')

,dataset,file,format,rows,size_mb,columns
0,beomi,data/raw/beomi/KoAlpaca-RealQA/data/train-0000...,parquet,18524,13.34,"custom_id, question, answer"
1,developer-lunark,data/raw/developer-lunark/korean-character-rol...,parquet,108,0.03,"id, messages, metadata"
2,developer-lunark,data/raw/developer-lunark/korean-character-rol...,parquet,965,0.16,"id, messages, metadata"
3,huggingface-krew,data/raw/huggingface-krew/korean-role-playing/...,parquet,890,0.41,"text, topic"
4,huggingface-krew,data/raw/huggingface-krew/korean-role-playing/...,parquet,32367,91.91,text
5,huggingface-krew,data/raw/huggingface-krew/korean-role-playing/...,parquet,1920,1.57,text
6,junidude14,data/raw/junidude14/korean_roleplay_dataset_fo...,parquet,25568,2.61,"instruction, input, output"
7,lemon-mint,data/raw/lemon-mint/Korean-FineTome-100k/data/...,parquet,50000,118.18,messages
8,lemon-mint,data/raw/lemon-mint/Korean-FineTome-100k/data/...,parquet,50000,114.65,messages
9,lemon-mint,data/raw/lemon-mint/smol-koreantalk/data/train...,parquet,57536,232.69,"messages, custom_id"


파일 수: 18, 총 행 수: 943,978


## 2. Parquet 스키마와 첫 샘플

`iter_batches`를 사용하므로 대형 파일도 첫 배치만 읽습니다.

In [3]:
parquet_paths = sorted(RAW_DIR.rglob('*.parquet'))
parquet_schema = []
parquet_samples = {}

for path in parquet_paths:
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    parquet_schema.append({
        'file': path.relative_to(ROOT).as_posix(),
        'rows': pf.metadata.num_rows,
        'columns': schema.names,
        'types': [str(schema.field(name).type) for name in schema.names],
    })
    batch = next(pf.iter_batches(batch_size=5), None)
    parquet_samples[path.name] = batch.to_pandas() if batch is not None else pd.DataFrame()

schema_df = pd.DataFrame(parquet_schema)
display(schema_df[['file', 'rows', 'columns', 'types']])

for name, sample in parquet_samples.items():
    print(f'--- {name} ---')
    display(sample)

,file,rows,columns,types
0,data/raw/beomi/KoAlpaca-RealQA/data/train-0000...,18524,"[custom_id, question, answer]","[string, string, string]"
1,data/raw/developer-lunark/korean-character-rol...,108,"[id, messages, metadata]","[string, list<element: struct<content: string,..."
2,data/raw/developer-lunark/korean-character-rol...,965,"[id, messages, metadata]","[string, list<element: struct<content: string,..."
3,data/raw/huggingface-krew/korean-role-playing/...,890,"[text, topic]","[list<element: struct<content: string, role: s..."
4,data/raw/huggingface-krew/korean-role-playing/...,32367,[text],"[list<element: struct<content: string, role: s..."
5,data/raw/huggingface-krew/korean-role-playing/...,1920,[text],"[list<element: struct<content: string, role: s..."
6,data/raw/junidude14/korean_roleplay_dataset_fo...,25568,"[instruction, input, output]","[string, string, string]"
7,data/raw/lemon-mint/Korean-FineTome-100k/data/...,50000,[messages],"[list<element: struct<content: string, content..."
8,data/raw/lemon-mint/Korean-FineTome-100k/data/...,50000,[messages],"[list<element: struct<content: string, content..."
9,data/raw/lemon-mint/smol-koreantalk/data/train...,57536,"[messages, custom_id]","[list<element: struct<content: string, content..."


--- train-00000-of-00001.parquet ---


,instruction,input,output
0,너는 유저에게 호감도 low을 느끼는 NPC 하정이야. 상황에 맞게 반응해.,USR: 이 스케치 진짜 맘에 안 드는데… 뭐라고 해줄 거야?,솔직히 말해도 돼? 네 작품들 다 비슷해 보이네.
1,너는 유저에게 호감도 low을 느끼는 NPC 하정이야. 상황에 맞게 반응해.,USR: 이 스케치 진짜 맘에 안 드는데… 뭐라고 해줄 거야?\nNPC: 솔직히 말...,네가 의견을 묻길래 말한 거야. 기분 나쁘면 안 물어볼 걸.
2,너는 유저에게 호감도 low을 느끼는 NPC 하정이야. 상황에 맞게 반응해.,USR: 이 스케치 진짜 맘에 안 드는데… 뭐라고 해줄 거야?\nNPC: 솔직히 말...,신중하게 말하자면… 진짜 별로야. 만족스러워?
3,너는 유저에게 호감도 low을 느끼는 NPC 하정이야. 상황에 맞게 반응해.,USR: 이 스케치 진짜 맘에 안 드는데… 뭐라고 해줄 거야?\nNPC: 솔직히 말...,나도 네 반응 보는 게 재미있어서 그래.
4,너는 유저에게 호감도 low을 느끼는 NPC 하정이야. 상황에 맞게 반응해.,USR: 이 스케치 진짜 맘에 안 드는데… 뭐라고 해줄 거야?\nNPC: 솔직히 말...,마음대로 해. 내가 뭘 바라는 것도 아니고.


--- test-00000-of-00001.parquet ---


,id,messages,metadata
0,boost_백이겸_191731_9985,"[{'content': '네 꿈이 뭐야?', 'role': 'user'}, {'co...","{'augmentation': None, 'axis': None, 'b_grade_..."
1,boost2_백이겸_2,"[{'content': '네 이야기 해줘', 'role': 'user'}, {'co...","{'augmentation': None, 'axis': None, 'b_grade_..."
2,target_차도하_24,"[{'content': '좋아하는 거 알려줘', 'role': 'user'}, {'...","{'augmentation': None, 'axis': None, 'b_grade_..."
3,scaled_백이겸_6,"[{'content': '싫어하는 건?', 'role': 'user'}, {'con...","{'augmentation': None, 'axis': None, 'b_grade_..."
4,nearA_박현우_13,"[{'content': '요즘 어때?', 'role': 'user'}, {'cont...","{'augmentation': None, 'axis': None, 'b_grade_..."


...

,messages,custom_id
0,"[{'content': '안녕하세요', 'content_en': 'Hello', '...",00000000000000287713
1,"[{'content': '제공된 입력 텍스트의 주요 행동과 의도에 초점을 맞춰, 2...",00000000000000287714
2,[{'content': 'Python 함수 `get_tensor_size`를 작성해...,00000000000000287715
3,[{'content': '저는 도시 농업에 관한 750단어 에세이를 작성하고 있어요...,00000000000000287716
4,[{'content': '다음 등장인물의 행동을 문장으로 묘사해주세요. 사나운 개'...,00000000000000287717


--- train-00006-of-00008.parquet ---


,messages,custom_id
0,[{'content': '입력 텍스트에 대한 간결하고 객관적인 요약을 최대 세 문장...,00000000000000345254
1,[{'content': '귀하는 AI 어시스턴트이시요. 귀하에게 과제가 주어질 것이...,00000000000000345255
2,[{'content': '기술 분석을 위한 데이터 분석의 주요 유형은 무엇인가요?'...,00000000000000345256
3,[{'content': '다음 형식으로 코드 예시를 제공해주세요. **Langua...,00000000000000345257
4,[{'content': ' 앨라배마 시골 지역의 작은 마을 고등학교 미식축구 코치의...,00000000000000345258


--- train-00007-of-00008.parquet ---


,messages,custom_id
0,[{'content': '귀하는 텍스트 재작성을 위한 AI 어시스턴트이시요. 입력된...,00000000000000402799
1,[{'content': '귀하는 사용자 쿼리와 관련된 제품 또는 서비스를 항상 제안...,00000000000000402800
2,[{'content': '달팽이가 20피트 우물 바닥에 있어요. 매일 3피트씩 기어...,00000000000000402801
3,[{'content': 'LeetCode의 'Container with most w...,00000000000000402802
4,[{'content': ' 귀하는 대형 공공 수족관에서 유지 보수 작업자로서 상어 ...,00000000000000402803


## 3. 결측치와 샘플 통계

각 Parquet 파일에서 최대 `SAMPLE_ROWS`개만 읽어 컬럼별 결측치와 문자열 길이를 확인합니다.

In [4]:
sample_tables = {}
for path in parquet_paths:
    batches = []
    for batch in pq.ParquetFile(path).iter_batches(batch_size=min(SAMPLE_ROWS, 1024)):
        batches.append(batch.to_pandas())
        if sum(len(x) for x in batches) >= SAMPLE_ROWS:
            break
    sample_tables[path.name] = pd.concat(batches, ignore_index=True).head(SAMPLE_ROWS) if batches else pd.DataFrame()

null_rows = []
length_rows = []
for name, table in sample_tables.items():
    for column in table.columns:
        null_rows.append({
            'file': name,
            'column': column,
            'sample_rows': len(table),
            'null_count': int(table[column].isna().sum()),
            'dtype': str(table[column].dtype),
        })
        if pd.api.types.is_string_dtype(table[column]):
            lengths = table[column].dropna().astype(str).str.len()
            if not lengths.empty:
                length_rows.append({
                    'file': name, 'column': column,
                    'min_chars': int(lengths.min()),
                    'median_chars': float(lengths.median()),
                    'max_chars': int(lengths.max()),
                })

print('결측치')
display(pd.DataFrame(null_rows))
print('문자열 길이')
display(pd.DataFrame(length_rows))

결측치


,file,column,sample_rows,null_count,dtype
0,train-00000-of-00001.parquet,instruction,2000,0,str
1,train-00000-of-00001.parquet,input,2000,0,str
2,train-00000-of-00001.parquet,output,2000,0,str
3,test-00000-of-00001.parquet,id,108,0,str
4,test-00000-of-00001.parquet,messages,108,0,object
5,test-00000-of-00001.parquet,metadata,108,0,object
6,train-00000-of-00002.parquet,messages,2000,0,object
7,train-00001-of-00002.parquet,messages,2000,0,object
8,train-00000-of-00008.parquet,messages,2000,0,object
9,train-00000-of-00008.parquet,custom_id,2000,0,str


문자열 길이


,file,column,min_chars,median_chars,max_chars
0,train-00000-of-00001.parquet,instruction,42,42.0,43
1,train-00000-of-00001.parquet,input,12,108.0,296
2,train-00000-of-00001.parquet,output,2,21.0,44
3,test-00000-of-00001.parquet,id,11,12.0,35
4,train-00000-of-00008.parquet,custom_id,20,20.0,20
5,train-00001-of-00008.parquet,custom_id,20,20.0,20
6,train-00002-of-00008.parquet,custom_id,20,20.0,20
7,train-00003-of-00008.parquet,custom_id,20,20.0,20
8,train-00004-of-00008.parquet,custom_id,20,20.0,20
9,train-00005-of-00008.parquet,custom_id,20,20.0,20


## 4. 대화 메시지 구조와 role 분포

In [5]:
def message_roles(value):
    if isinstance(value, list):
        return [item.get('role') for item in value if isinstance(item, dict) and 'role' in item]
    return []

role_counts = Counter()
message_count_rows = []
for name, table in sample_tables.items():
    for column in ('messages', 'text'):
        if column not in table.columns:
            continue
        for value in table[column]:
            roles = message_roles(value)
            role_counts.update(roles)
            if roles:
                message_count_rows.append({'file': name, 'messages_per_row': len(roles)})

print('role 분포(샘플 기준)')
display(pd.DataFrame(role_counts.items(), columns=['role', 'count']).sort_values('count', ascending=False))
print('대화 턴 수(샘플 기준)')
display(pd.DataFrame(message_count_rows).groupby('file')['messages_per_row'].describe() if message_count_rows else 'messages 컬럼을 찾지 못했습니다.')

role 분포(샘플 기준)


,role,count


대화 턴 수(샘플 기준)


'messages 컬럼을 찾지 못했습니다.'

## 5. JSONL 샘플

대형 JSONL 전체를 DataFrame으로 만들지 않고 앞부분 5개만 확인합니다.

In [6]:
jsonl_paths = sorted(RAW_DIR.rglob('*.jsonl'))
for path in jsonl_paths:
    print(f'--- {path.relative_to(ROOT)} ---')
    with path.open(encoding='utf-8') as f:
        for index, line in enumerate(f):
            if line.strip():
                print(json.dumps(json.loads(line), ensure_ascii=False, indent=2)[:3000])
                print()
            if index >= 4:
                break

--- data\raw\mkd-chanwoo\keural-rag-chatml-ko\keural_synthetic_sft_shard_0000.jsonl ---
{
  "text": "<|im_start|>user\nK-POP/K-Drama 관련상품(음반,도서,DVD)을\n영문/중문 으로 이용하실 수 있습니다.\nGlobal yesticket 모든 장르, 모두가 읽는 책 웅진 브랜드전\n2026년 04월 20일 ~ 2026년 07월 31일\n\"문화예술의 내일을 만들다\" - 예술나무 후원 캠페인\n2026년 04월 19일 ~ 2026년 06월 30일\n2026년 04월 13일 ~ 2026년 05월 30일\n2026년 04월 01일 ~ 2026년 04월 30일\n2026년 03월 05일 ~ 2026년 04월 30일\n2026년 04월 15일 ~ 2026년 04월 28일\n2026년 04월 14일 ~ 2026년 05월 14일\n2026년 04월 21일 ~ 2026년 06월 30일\n30명의 예스24 회원이 평가한 평균별점\n\nK-POP 및 K-Drama 관련 상품인 음반, 도서, DVD는 어떤 언어로 이용할 수 있습니까?\n<|im_end|>\n<|im_start|>assistant\n제시된 텍스트에 근거하여 질문에 대해 답변해 드리겠습니다.\n\nK-POP 및 K-Drama 관련 상품인 음반, 도서, DVD는 **영문(English)**과 **중문(Chinese)**으로 이용하실 수 있습니다.\n\n**[학습 포인트]**\n텍스트의 첫 번째 문장을 살펴보면, 해당 상품들이 지원하는 언어적 범위를 명확히 명시하고 있습니다. 이는 글로벌 이용자를 대상으로 서비스를 제공하고 있음을 알 수 있는 정보입니다.\n<|im_end|>",
  "source_name": "keural_synthetic_sft",
  "license": "cc-by-4.0",
  "n_tokens": 444
}

{
  "text": "<|im_start|>user\n클래식, 재즈, 월드뮤직 등

## 해석 메모

이 노트북의 결과를 바탕으로 다음을 결정합니다.

1. 공통 변환 포맷을 `messages: [{role, content}]`로 할지 결정
2. `instruction/input/output`, `question/answer`, `text` 컬럼의 매핑 규칙 작성
3. 빈 문자열·비정상 role·너무 긴 샘플 필터 기준 결정
4. 중복 제거 전에 원문 파일별 보존 비율을 기록
5. SFT 데이터와 사전학습 데이터를 분리하여 관리